In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

## Описание данных

**Целевая переменная:** `SeriousDlqin2yrs` — был ли у клиента просрочка 90+ дней или хуже за последние 2 года.

| Признак | Описание | Тип данных |
|---------|----------|-------------|
| `SeriousDlqin2yrs` | Просрочка 90+ дней или хуже за последние 2 года | Y/N (целевой) |
| `RevolvingUtilizationOfUnsecuredLines` | Отношение общего остатка по кредитным картам и персональным кредитным линиям (кроме недвижимости и рассрочки) к сумме кредитных лимитов | доля |
| `age` | Возраст заёмщика в годах | целое число |
| `NumberOfTime30-59DaysPastDueNotWorse` | Количество раз, когда заёмщик допускал просрочку 30–59 дней (но не более) за последние 2 года | целое число |
| `DebtRatio` | Отношение ежемесячных выплат по долгам, алиментов, расходов на жизнь к валовому месячному доходу | доля |
| `MonthlyIncome` | Ежемесячный доход | вещественное число |
| `NumberOfOpenCreditLinesAndLoans` | Количество открытых кредитов (например, автокредит, ипотека) и кредитных линий (например, кредитные карты) | целое число |
| `NumberOfTimes90DaysLate` | Количество раз, когда заёмщик допускал просрочку 90+ дней | целое число |
| `NumberRealEstateLoansOrLines` | Количество ипотечных кредитов и кредитов под недвижимость, включая возобновляемые кредитные линии под залог недвижимости | целое число |
| `NumberOfTime60-89DaysPastDueNotWorse` | Количество раз, когда заёмщик допускал просрочку 60–89 дней (но не более) за последние 2 года | целое число |
| `NumberOfDependents` | Количество иждивенцев в семье (не включая самого заёмщика) — супруг(а), дети и т.д. | целое число |


In [2]:
train_file = 'data.csv'

In [50]:
df = pd.read_csv(train_file)
df.head()

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs
0,0.114987,62,0,1841.000000,NaN,5,0,1,0,2.0,0
1,0.008705,73,0,0.498553,3800.0,6,0,1,0,0.0,0
2,0.214501,32,0,0.211999,3716.0,8,0,0,0,2.0,0
3,1.000000,60,0,118.000000,NaN,5,0,0,0,0.0,0
4,0.230493,60,0,1.017328,3000.0,10,0,1,0,0.0,0


In [49]:
TARGET_FEATURE = "SeriousDlqin2yrs"
X = df.drop(TARGET_FEATURE, axis=1)
y = df[TARGET_FEATURE]

## Задание

Обучить линейную модель предсказания целевой переменной. Оценить качество. Считается, что дать кредит человеку, который допустит просрочку, для банка в 5 раз дороже, чем не дать платёжеспособному.

Ноутбук должен быть рабочим. Также должна быть реализована функция predict(model, data_path), возвращающая предсказанные значения. 

In [22]:
def predict(model, data_path):

    df = pd.read_csv(data_path)
    
    X = df.drop("SeriousDlqin2yrs", axis=1)

    return model.predict(X)


In [23]:
clf = DummyClassifier(strategy="most_frequent")
clf.fit(X, y)

preds = predict(clf, train_file)
print(preds)
print(accuracy_score(y, preds))

[0 0 0 ... 0 0 0]
0.9331583333333333


LETSGOOO

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [11]:
import catboost

clf = catboost.CatBoostClassifier()
clf.fit(X_train, y_train, verbose=False)

CatBoostClassifier()

In [13]:
y_pred = clf.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix 

res = confusion_matrix(y_test, y_pred, labels=[0,1])
tn, fp, fn, tp = res.ravel().tolist()
res

array([[36550,   346],
       [ 2169,   535]])

Выдать тому, кто просрочит - дороже, чем не выдать. Соответственно, метрика должна учитывать FN важнее, чем FP - важнее определить человека, который просрочит, чем пропустить.

Рассматривая метрики
- Precission = TP / (TP + FP),
- Recall = TP / (TP + FN)

получается, что Recall важнее для нас. Согласно https://education.yandex.ru/handbook/ml/article/metriki-klassifikacii-i-regressii#tochnost-i-polnota можно использовать F-beta score для совместного их рассмотрения с учетом различной важности

Можно придумать свою метрику качества, например, 1 - (5 * FN + FP) / (TP + FP + TN + FN)

In [36]:
from sklearn.metrics import accuracy_score, fbeta_score

print(f"accuracy: {accuracy_score(y_test, y_pred)}")

FN_IMPORTANCE = 5 # times 
print(f"f-beta(beta={FN_IMPORTANCE}): {fbeta_score(y_test, y_pred, beta=FN_IMPORTANCE)}")

print(f"custom: {1 - (FN_IMPORTANCE * fn + fp) / (tp + fp + tn + fn):.7f}")

accuracy: 0.936489898989899
f-beta(beta=5): 0.20312203384880478
custom: 0.7173990


Попробуем почистить данные

In [51]:
df.describe()

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs
count,120000.000000,120000.000000,120000.000000,120000.000000,9.632500e+04,120000.000000,120000.000000,120000.000000,120000.000000,116872.000000,120000.000000
mean,6.128916,52.287842,0.420075,352.271245,6.651507e+03,8.465692,0.264942,1.018167,0.239858,0.757333,0.066842
std,253.361490,14.771274,4.182138,2093.709509,1.454118e+04,5.161693,4.158242,1.133055,4.144569,1.115337,0.249749
min,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.029593,41.000000,0.000000,0.175330,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.153318,52.000000,0.000000,0.366194,5.390000e+03,8.000000,0.000000,1.000000,0.000000,0.000000,0.000000
75%,0.557832,63.000000,0.000000,0.860833,8.238000e+03,11.000000,0.000000,2.000000,0.000000,1.000000,0.000000
max,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000,1.000000


In [52]:
df.isna().sum()

RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           23675
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3128
SeriousDlqin2yrs                            0
dtype: int64

In [58]:
X = df.drop(TARGET_FEATURE, axis=1)
y = df[TARGET_FEATURE]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

In [ ]:
df = X_train.assign(**{TARGET_FEATURE: y_train})
df

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs
95474,0.691789,41,0,0.238621,3800.0,4,0,0,0,3.0,0
9635,1.000000,24,0,0.000000,1530.0,2,0,0,0,0.0,0
23237,0.884888,49,0,1244.000000,5400.0,4,0,0,0,0.0,0
112145,0.223785,71,0,0.032248,3100.0,1,0,0,0,0.0,0
42504,0.037781,69,0,25.000000,5400.0,4,0,0,0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...
4847,0.780261,69,0,4708.000000,5400.0,21,0,1,0,0.0,0
71816,0.097395,29,0,0.427075,6300.0,11,0,2,0,0.0,0
57589,0.009842,72,0,0.354077,4659.0,23,0,2,0,0.0,0
77260,1.000000,58,0,0.135550,2736.0,1,0,0,0,0.0,1


In [60]:
df = df.drop_duplicates()
df.fillna(
    {
        "NumberOfDependents": df["NumberOfDependents"].mean(),
        "MonthlyIncome": df["MonthlyIncome"].median(), # more fair mean
    },
    inplace=True,
)
df.describe()

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs
count,53726.000000,53726.000000,53726.000000,53726.000000,5.372600e+04,53726.000000,53726.000000,53726.000000,53726.000000,53726.000000,53726.000000
mean,7.433684,52.365019,0.404292,357.766143,6.405767e+03,8.475785,0.248316,1.019413,0.222071,0.755880,0.066932
std,303.624642,14.768754,3.968098,2348.127323,1.475760e+04,5.147680,3.943180,1.142770,3.926978,1.100711,0.249907
min,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.029440,41.000000,0.000000,0.177050,3.900000e+03,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.151197,52.000000,0.000000,0.368925,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000,0.000000
75%,0.557471,63.000000,0.000000,0.883356,7.384000e+03,11.000000,0.000000,2.000000,0.000000,1.000000,0.000000
max,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000,1.000000


In [62]:
X_train = df.drop(TARGET_FEATURE, axis=1)
y_train = df[TARGET_FEATURE]

In [65]:
clf = catboost.CatBoostClassifier()
clf.fit(X_train, y_train, verbose=False)

CatBoostClassifier()

In [66]:
X_test.fillna(
    {
        "NumberOfDependents": df["NumberOfDependents"].mean(),
        "MonthlyIncome": df["MonthlyIncome"].median(), # more fair mean
    },
    inplace=True,
)
y_pred = clf.predict(X_test)

In [67]:
res = confusion_matrix(y_test, y_pred, labels=[0,1])
tn, fp, fn, tp = res.ravel().tolist()
res

array([[24425,   267],
       [ 1440,   331]])

In [68]:
print(f"accuracy: {accuracy_score(y_test, y_pred)}")

FN_IMPORTANCE = 5 # times 
print(f"f-beta(beta={FN_IMPORTANCE}): {fbeta_score(y_test, y_pred, beta=FN_IMPORTANCE)}")

print(f"custom: {1 - (FN_IMPORTANCE * fn + fp) / (tp + fp + tn + fn):.7f}")

accuracy: 0.935494841854665
f-beta(beta=5): 0.19178570632674435
custom: 0.7178324
